# Module 13: The Rosetta Stone
## RL and Active Inference Are Closer Than You Think

**Spinning Up in Active Inference** | Notebook 13 of 16

---

Throughout this curriculum we have presented Reinforcement Learning and Active Inference as distinct frameworks. In this module we reveal their deep formal connections. The two are not rival theories -- they are different *decompositions* of the same underlying problem, and under specific conditions they produce identical behavior.

By the end of this notebook you will:
1. Have a comprehensive Rosetta Stone table mapping RL concepts to AIF concepts
2. Run the *same* environment with both an RL agent (MBV) and an AIF agent
3. Compare learned representations: Q-values vs. negative EFE
4. Prove the mathematical equivalence under appropriate limits
5. Understand where the frameworks *genuinely differ* (epistemic value)

**Key references:**
- Millidge, Tschantz & Buckley (2020). *Whence the Expected Free Energy?*
- Sajid, Ball, Parr & Friston (2021). *Active Inference: Demystified and Compared.*
- Da Costa et al. (2020). *Active Inference on Discrete State-Spaces: A Synthesis.*

**Prerequisites:** Modules 1-12, familiarity with Q-learning and EFE.
**Time:** ~60 minutes.

## 13.1 Correspondences and Their Limits

| RL concept | AIF concept | Relationship |
|---|---|---|
| Reward utility $r(o)$ | Log preference $C(o)$ | Set $C=r$ to match utility; normalized preferences are proportional to $\exp(r)$ |
| One-step expected reward | Negative EFE | $-G=E[C]+I(s;o)$; equality requires zero information gain |
| Softmax policy | $\operatorname{softmax}(-\gamma G+\ln E)$ | Same algebraic form; values need not be the same |
| Transition model | $B(s'|s,a)$ | Same probability semantics, possibly different array axes |
| Observation model | $A(o|s)$ | Both can represent partial observability |
| Model learning | Learning $A,B$ | Both estimate a world model from data |

These correspondences are conditional, not a general equivalence of Bellman values and EFE. RL can also use information objectives. Discounting and finite planning horizons are distinct choices.

In [ ]:
# -- Setup ------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# neuro-nav: RL agents and environments
from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate
from neuronav.agents.mb_agents import MBV

# Active Inference (alf)
from alf.generative_model import GenerativeModel
from alf.free_energy import (
    variational_free_energy,
    expected_free_energy_decomposed,
    EFEDecomposition,
)
from alf.policy import select_action
from alf.benchmarks.neuronav_wrappers import (
    GridEnvToGenerativeModel,
    NeuroNavRunner,
)

# Curriculum plotting utilities
import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_value_heatmap, plot_learning_curve,
    dual_perspective_box, COLORS,
)

np.random.seed(42)
print("Imports successful -- let's build the Rosetta Stone!")

## 13.2 Same Environment, Two Frameworks

We use the **Four Rooms** grid from neuro-nav -- a classic navigation benchmark. We will set up:
1. An **MBV** (Model-Based Value) RL agent that learns a transition model and computes value functions
2. An **AIF** agent via `GridEnvToGenerativeModel` that uses the same transition structure

Both agents solve the same problem: reach the goal. The question is *how* their internal representations compare.

In [ ]:
# -- Create the shared environment ------------------------------------------
env = GridEnv(
    template=GridTemplate.four_rooms,
    size=GridSize.small,
    obs_type=GridObservation.index,
)
obs = env.reset()
goal_pos = list(env.objects['rewards'].keys())[0]

grid_size = env.grid_size
num_states = grid_size ** 2
num_actions = 4  # N, E, S, W

print(f"Grid: {grid_size}x{grid_size} = {num_states} states")
print(f"Actions: 4 (N, E, S, W)")
print(f"Goal: {goal_pos}")

# -- Set up the MBV RL agent -----------------------------------------------
mbv_agent = MBV(
    state_size=num_states,
    action_size=num_actions,
    lr=0.1,
    gamma=0.95,
)

# -- Set up the AIF agent via wrapper ---------------------------------------
gm = GridEnvToGenerativeModel(env, reward_magnitude=3.0, T=1)

print(f"\nMBV agent: state_size={num_states}, action_size={num_actions}")
print(f"AIF model: A shape={gm.A[0].shape}, B shape={gm.B[0].shape}")
print(f"AIF model: C (preferences) has {np.count_nonzero(gm.C[0])} non-zero entries")

In [ ]:
# -- Train both agents on the same environment for the same number of episodes

n_episodes = 150
max_steps = 300

# RL agent: MBV learning curves
mbv_rewards = []
mbv_steps = []

for ep in range(n_episodes):
    obs = env.reset()
    total_reward = 0.0
    done = False
    steps = 0

    while not done and steps < max_steps:
        action = mbv_agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        mbv_agent.update((obs, action, next_obs, reward, done))
        obs = next_obs
        total_reward += reward
        steps += 1

    mbv_rewards.append(total_reward)
    mbv_steps.append(steps)

# AIF agent: use NeuroNavRunner for fair comparison
runner = NeuroNavRunner(
    gm=gm,
    env=env,
    aif_gamma=4.0,
    max_steps=max_steps,
    seed=42,
)

aif_rewards = []
aif_steps_list = []

for ep in range(n_episodes):
    result = runner._run_aif_episode()
    aif_rewards.append(result['total_reward'])
    aif_steps_list.append(result['steps'])

print(f"MBV  -- Mean reward (last 20): {np.mean(mbv_rewards[-20:]):.3f}, "
      f"mean steps: {np.mean(mbv_steps[-20:]):.1f}")
print(f"AIF  -- Mean reward (last 20): {np.mean(aif_rewards[-20:]):.3f}, "
      f"mean steps: {np.mean(aif_steps_list[-20:]):.1f}")

In [ ]:
# -- Plot learning curves side by side --------------------------------------

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plot_learning_curve(
    {"MBV (RL)": mbv_rewards, "AIF": aif_rewards},
    ylabel="Episode Reward",
    title="Learning Curves: Reward",
    window=10,
    ax=ax1,
)

plot_learning_curve(
    {"MBV (RL)": mbv_steps, "AIF": aif_steps_list},
    ylabel="Steps to Goal",
    title="Learning Curves: Episode Length",
    window=10,
    ax=ax2,
)

plt.tight_layout()
plt.show()

print("Note: MBV learns its transition model from experience (gradual improvement).")
print("AIF has the true transition model from the start (GridEnvToGenerativeModel),")
print("so it may perform well from episode 1 -- like Tolman's rats with a cognitive map.")

## 13.3 Comparing Learned Representations

Now let us look *inside* each agent. The MBV agent has learned:
- A **transition model** T(s'|s,a) from experience
- **Q-values** Q(s,a) via value iteration on the learned model

The AIF agent has:
- A **B-matrix** P(s'|s,a) -- the true transition model from the environment
- **Negative EFE** -G(a) for each action at each state

If the Rosetta Stone is correct, these should look similar.

In [ ]:
# -- Compare transition models: MBV's learned T vs AIF's B-matrix -----------

# MBV stores its learned model in mbv_agent.T with shape (num_actions, num_states, num_states)
# The B-matrix from AIF is gm.B[0] with shape (num_states, num_states, num_actions)
B_aif = gm.B[0]  # True transition matrix

# MBV's transition model: T[a, s, s'] = P(s'|s,a)
T_mbv = mbv_agent.T  # shape: (num_actions, num_states, num_states)

# Visualize transition matrices for action 1 (East)
action_idx = 1  # East
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Show a subset (first 25x25 states) for readability
n_show = min(25, num_states)

# MBV T: shape (num_actions, num_states, num_states) -> T[action_idx, :n_show, :n_show]
im1 = ax1.imshow(T_mbv[action_idx, :n_show, :n_show], cmap='viridis', aspect='auto')
ax1.set_title(f"MBV Learned T-matrix (action=East)\nFirst {n_show} states", fontsize=12)
ax1.set_xlabel("From state")
ax1.set_ylabel("To state")
plt.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(B_aif[:n_show, :n_show, action_idx], cmap='viridis', aspect='auto')
ax2.set_title(f"AIF B-matrix (action=East)\nFirst {n_show} states", fontsize=12)
ax2.set_xlabel("From state")
ax2.set_ylabel("To state")
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.suptitle("Rosetta Stone: T(s'|s,a) = B(s'|s,a)\nIdentical semantics, different names",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# -- Compare value maps: MBV Q-values vs AIF negative EFE -------------------

# MBV: V(s) = max_a Q(s,a)
# Q shape is (num_actions, num_states) — max over axis=0 gives V(s)
q_table_mbv = mbv_agent.Q  # shape: (num_actions, num_states)
v_mbv = np.max(q_table_mbv, axis=0)

# AIF: V(s) ~ -min_a G(a|s) = max_a [-G(a|s)]
# Compute -G for each state assuming the agent is at that state
A = gm.A[0]
B = gm.B[0]
C = gm.C[0]

v_aif = np.zeros(num_states)
for s in range(num_states):
    beliefs = np.zeros(num_states)
    beliefs[s] = 1.0

    neg_G_values = []
    for a in range(num_actions):
        decomp = expected_free_energy_decomposed(A, B, C, beliefs, a)
        neg_G_values.append(-decomp.G_total)

    v_aif[s] = max(neg_G_values)

# Plot side by side
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

plot_value_heatmap(v_mbv, grid_size, title="MBV: V(s) = max_a Q(s,a)",
                   cmap="inferno", goal_pos=goal_pos, ax=ax1)
plot_value_heatmap(v_aif, grid_size, title="AIF: max_a [-G(a|s)]",
                   cmap="inferno", goal_pos=goal_pos, ax=ax2)

# Correlation plot
# Normalize both to [0, 1] for comparison
v_mbv_norm = (v_mbv - v_mbv.min()) / (v_mbv.max() - v_mbv.min() + 1e-10)
v_aif_norm = (v_aif - v_aif.min()) / (v_aif.max() - v_aif.min() + 1e-10)

ax3.scatter(v_mbv_norm, v_aif_norm, alpha=0.4, s=20, color=COLORS['aif'])
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='y = x')
corr = np.corrcoef(v_mbv_norm, v_aif_norm)[0, 1]
ax3.set_xlabel("V_RL(s) [normalized]")
ax3.set_ylabel("-G_AIF(s) [normalized]")
ax3.set_title(f"Correlation: r = {corr:.3f}")
ax3.legend()
ax3.set_aspect('equal')

plt.suptitle("Rosetta Stone: V(s) vs. -G(s)\nDifferent objectives and horizons; compare with care",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 13.4 When Does the Policy Match Expected-Reward Maximization?

$$G(a)=-E[C(o')]-I(s';o').$$

When $A=I$, conditional observation entropy is zero, but information gain is **$H[Q(s'|a)]$**, which can be positive with stochastic transitions or uncertain starting beliefs.

If the current state and transitions are deterministic, $Q(s'|a)$ is a point mass and information gain is zero. Setting $C=r$, using the same horizon and uniform habits, then gives the same softmax policy as one-step expected-reward maximization at $\gamma=1/\tau$.

An information term that is constant across actions also cancels from the policy, though it still shifts the scores. Setting $C=\log r$ changes the utility: expected log reward is generally not expected reward, and action rankings can differ.

In [ ]:
# Perfect observations still provide information under stochastic transitions.
num_s, num_a = 5, 3
A_ident = np.eye(num_s)
rng = np.random.default_rng(42)
B_test = rng.dirichlet(np.ones(num_s), size=(num_s, num_a)).transpose(2, 0, 1)
rewards = np.array([0.1, 0.5, 1.0, 0.3, 2.0])
C_preferences = rewards.copy()  # log preferences equal reward utility
beliefs = np.eye(num_s)[0]
print("Action | expected reward | information gain | -G")
for a in range(num_a):
    decomp = expected_free_energy_decomposed(A_ident, B_test, C_preferences, beliefs, a)
    predicted_s = B_test[:, 0, a]
    expected_reward = predicted_s @ rewards
    predictive_entropy = -np.sum(predicted_s * np.log(predicted_s))
    np.testing.assert_allclose(decomp.epistemic, predictive_entropy, atol=1e-10)
    np.testing.assert_allclose(-decomp.G_total, expected_reward + predictive_entropy, atol=1e-10)
    print(a, expected_reward, decomp.epistemic, -decomp.G_total)
print("A=I removes observation ambiguity, not uncertainty about the next state.")

In [ ]:
# Deterministic next states give zero information gain from a known start.
B_deterministic = np.zeros_like(B_test)
for a, destination in enumerate([0, 2, 4]):
    B_deterministic[destination, :, a] = 1.0

def softmax(x):
    weights = np.exp(x - np.max(x))
    return weights / weights.sum()

gamma_aif = 4.0
tau_rl = 1.0 / gamma_aif
q_values_rl = np.array([B_deterministic[:, 0, a] @ rewards for a in range(num_a)])
G_values = np.array([
    expected_free_energy_decomposed(A_ident, B_deterministic, C_preferences, beliefs, a).G_total
    for a in range(num_a)
])
pi_rl = softmax(q_values_rl / tau_rl)
pi_aif = softmax(-gamma_aif * G_values)
np.testing.assert_allclose(-G_values, q_values_rl, atol=1e-10)
np.testing.assert_allclose(pi_rl, pi_aif, atol=1e-10)
print("Matched deterministic case:", pi_rl, pi_aif)
print("This equality depends on matched utility, beliefs, horizon, and habits.")

## 13.5 Information Depends on Both the Sensor and Predictive Beliefs

A perfect sensor can reveal an uncertain future state; a partially observable sensor can reveal nothing if its output is independent of the state. Compare the actual mutual information, rather than using full versus partial observability as a shortcut.

The following examples compare the same stochastic transition model with perfect and imperfect sensors. Their utility vectors differ with the observation spaces, so this is a decomposition illustration, not a controlled performance comparison.

In [ ]:
# -- Epistemic value: the key difference ------------------------------------

# Same 5-state environment, but now PARTIALLY OBSERVABLE
# Two observations map to overlapping states:
#   obs 0: mostly state 0 or 1 (ambiguous)
#   obs 1: mostly state 2
#   obs 2: mostly state 3 or 4 (ambiguous)

num_obs = 3
A_partial = np.array([
    [0.7, 0.6, 0.1, 0.05, 0.05],  # obs 0
    [0.2, 0.2, 0.8, 0.15, 0.15],  # obs 1
    [0.1, 0.2, 0.1, 0.80, 0.80],  # obs 2
])
# Normalize columns
A_partial = A_partial / A_partial.sum(axis=0, keepdims=True)

# Use same B and C from before
# But C must match observation space now
C_partial = np.array([0.0, 1.0, 2.0])  # prefer obs 2 (which indicates states 3-4)

# Uncertain starting beliefs
beliefs_uncertain = np.ones(num_s) / num_s

print("EFE Decomposition: Partially Observable Environment")
print(f"{'Action':<8} {'G_total':>10} {'Pragmatic':>11} {'Epistemic':>11} "
      f"{'% Epistemic':>12}")
print("-" * 60)

for a in range(num_a):
    decomp = expected_free_energy_decomposed(
        A_partial, B_test, C_partial, beliefs_uncertain, a
    )
    total = abs(decomp.pragmatic) + abs(decomp.epistemic)
    pct_epist = abs(decomp.epistemic) / total * 100 if total > 0 else 0
    print(f"  {a}      {decomp.G_total:>10.4f} {decomp.pragmatic:>11.4f} "
          f"{decomp.epistemic:>11.4f} {pct_epist:>11.1f}%")

print("\n--- Now compare with fully observable (A = I) ---")
print(f"{'Action':<8} {'G_total':>10} {'Pragmatic':>11} {'Epistemic':>11}")
print("-" * 45)
for a in range(num_a):
    decomp = expected_free_energy_decomposed(
        A_ident, B_test, C_preferences, beliefs_uncertain, a
    )
    print(f"  {a}      {decomp.G_total:>10.4f} {decomp.pragmatic:>11.4f} "
          f"{decomp.epistemic:>11.4f}")

print("\nWith A=I and uncertain next states, epistemic value is predictive state entropy.")
print("An imperfect sensor can reveal some, all, or none of that information.")
print("Evaluate information gain under the predictive joint distribution.")

In [ ]:
# -- Visualize the decomposition --------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fully observable
pragmatic_fo = []
epistemic_fo = []
for a in range(num_a):
    d = expected_free_energy_decomposed(A_ident, B_test, C_preferences, beliefs_uncertain, a)
    pragmatic_fo.append(d.pragmatic)
    epistemic_fo.append(d.epistemic)

x_pos = np.arange(num_a)
axes[0].bar(x_pos - 0.15, pragmatic_fo, 0.3, label='Pragmatic (reward)',
            color=COLORS.get('rl', '#2196F3'))
axes[0].bar(x_pos + 0.15, epistemic_fo, 0.3, label='Epistemic (info gain)',
            color=COLORS.get('aif', '#4CAF50'))
axes[0].set_xlabel('Action')
axes[0].set_ylabel('Value')
axes[0].set_title('Fully Observable (A = I)\nInformation gain = predictive state entropy')
axes[0].set_xticks(x_pos)
axes[0].legend()
axes[0].axhline(0, color='k', linewidth=0.5)

# Partially observable
pragmatic_po = []
epistemic_po = []
for a in range(num_a):
    d = expected_free_energy_decomposed(
        A_partial, B_test, C_partial, beliefs_uncertain, a
    )
    pragmatic_po.append(d.pragmatic)
    epistemic_po.append(d.epistemic)

axes[1].bar(x_pos - 0.15, pragmatic_po, 0.3, label='Pragmatic (reward)',
            color=COLORS.get('rl', '#2196F3'))
axes[1].bar(x_pos + 0.15, epistemic_po, 0.3, label='Epistemic (info gain)',
            color=COLORS.get('aif', '#4CAF50'))
axes[1].set_xlabel('Action')
axes[1].set_ylabel('Value')
axes[1].set_title('Partially Observable (A != I)\nEpistemic value drives exploration')
axes[1].set_xticks(x_pos)
axes[1].legend()
axes[1].axhline(0, color='k', linewidth=0.5)

plt.suptitle('The Rosetta Stone Boundary: Where RL and AIF Diverge',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# -- Summary visualization: the spectrum from RL to AIF ---------------------

dual_perspective_box(
    rl_text=(
        "RL maximizes expected cumulative reward: E[sum_t gamma^t r_t].\n"
        "The policy is pi(a|s) = softmax(Q(s,a)/tau).\n"
        "Exploration requires external mechanisms (epsilon-greedy, UCB).\n"
        "The transition model T(s'|s,a) is learned from experience.\n"
        "Works best in fully observable, reward-rich environments."
    ),
    aif_text=(
        "AIF minimizes Expected Free Energy: G = -pragmatic - epistemic.\n"
        "The policy is pi(a) = softmax(-gamma*G + ln E).\n"
        "Exploration is intrinsic: epistemic value = information gain.\n"
        "The generative model (A, B, C, D) defines the agent's world.\n"
        "Naturally handles partial observability and sparse reward."
    ),
    title="RL vs. Active Inference: Same Math, Different Emphasis"
)

## 13.6 Exercise: State the Assumptions of Policy Equivalence

Prove $\operatorname{softmax}(Q/\tau)=\operatorname{softmax}(-\gamma G)$ when:

- $C=r$, with a common observation utility and a one-step horizon;
- the current state is known, transitions are deterministic, and $A=I$;
- habits are uniform and $\gamma=1/\tau$.

Then replace the deterministic transitions by `B_test`. Explain why the added predictive entropy can change action rankings. Finally, construct a pair of lotteries where maximizing expected reward and expected log reward choose different actions.

In [ ]:
for gamma in [0.5, 1.0, 2.0, 4.0, 8.0, 16.0]:
    pi_reward = softmax(gamma * q_values_rl)
    pi_efe = softmax(-gamma * G_values)
    np.testing.assert_allclose(pi_reward, pi_efe, atol=1e-10)
    print(f"gamma={gamma}: max difference={np.max(np.abs(pi_reward-pi_efe)):.3g}")

# A counterexample to replacing reward by log reward:
# certain payoff 2 versus an equal-probability lottery paying 0.1 or 10.
mean_reward = np.array([2.0, (0.1 + 10.0) / 2])
mean_log_reward = np.array([np.log(2.0), (np.log(0.1) + np.log(10.0)) / 2])
assert np.argmax(mean_reward) != np.argmax(mean_log_reward)
print("Reward and log-reward utilities can reverse the preferred action.")

## 13.7 Summary and Looking Forward

The Rosetta Stone reveals that RL and Active Inference are not competing theories but complementary perspectives on the same problem:

1. **With zero information gain and C=r**: negative EFE equals one-step expected reward. A=I alone is insufficient.
2. **With nonzero information gain**: EFE includes an additional state-information objective; this does not guarantee better reward performance.
3. **The policy structure is identical**: both use softmax over action values, with temperature/precision controlling exploration.
4. **The models are identical**: transition matrices T = B, observation models = A matrices.

The decomposition distinguishes preference satisfaction from information gain. Comparisons require matched models, data, utility, and planning assumptions.

### Extending the Rosetta Stone: Modern Architectures on Cognitive Tasks

The RL↔AIF comparison we demonstrated here used tabular agents on grid worlds. But neuroscience uses a much richer task battery — and modern architectures are revealing surprising things about learned representations:

- **Tiny RNNs** (Ji-An et al., [Nature 2025](https://www.nature.com/articles/s41586-024-08145-x)): 1–4 unit RNNs outperform 30+ classical RL/Bayesian models on reward-learning tasks when fit to individual subjects. The dynamics reveal continuously varying learning rates and state-dependent perseveration — strategies invisible to both classical RL and AIF model-fitting.
- **Transformers learn AIF-like gating**: Whittington et al. (arXiv 2024) showed that transformers trained on working memory tasks spontaneously learn input/output gating that mirrors the A-matrix's role in AIF — filtering what enters and exits the belief state.
- **Dynamical Similarity Analysis** ([Ostrow et al., NeurIPS 2023](https://arxiv.org/abs/2301.10197)): Koopman spectral methods can distinguish different *dynamic implementations* of the same computation across architectures — geometric methods (CKA, Procrustes) cannot. This could be used to compare RL vs. AIF agent internals at the level of dynamics, not just behavior.
- **NeuroGym** ([neurogym.github.io](https://github.com/neurogym/neurogym)) provides 30+ Gymnasium-compatible cognitive tasks (Go/No-Go, DawTwoStep, PerceptualDecisionMaking, DelayMatchSample, etc.) — the standard battery used in systems neuroscience labs. Running both RL and AIF agents on these tasks, with representation analysis, is the natural next step for the Rosetta Stone comparison.

### Open Directions

1. Run the Rosetta Stone comparison on **DawTwoStep-v0** from NeuroGym — the gold-standard task for distinguishing model-based from model-free strategies
2. Compare internal representations using **DSA** (Koopman spectral analysis) rather than just value maps
3. Train **continuous-time networks** (CfCs) on the same tasks — biological dynamics are continuous, and neither RNNs nor AIF's discrete-time formulation captures this
4. Build a **differentiable AIF agent** that can be trained end-to-end on NeuroGym tasks with gradient-based learning — bridging Modules 12 and 13

**Next:** [Module 14: Scaling with JAX](14_jax_scaling.ipynb) -- we take these ideas to hardware, using JAX's `jit`, `vmap`, and `grad` to run thousands of agents simultaneously.